
# 練習問題: 2D 熱伝導 (ラプラス方程式) をヤコビ反復で解く

## 目標

熱した板の**定常温度分布**を, 差分法 (5点ステンシル) のヤコビ反復で求める。
内部の二重ループを `collapse(2)` で並列化し, 収束判定に使う「1反復での最大更新量」を `reduction(max:diff)` で求める, という stencil 計算の典型パターンを身につける。

## 課題

`heat2d.cpp` (または `heat2d.f90`) は, L×L の板の上端 (行0) を温度 100, 残り3辺を 0 に固定し, 内部の各点を**上下左右4点の平均**で繰り返し更新する (ヤコビ法)。各反復で更新量が一番大きかった値 `diff` を求め, それが `tol` を下回ったら収束とみなす。

`TODO` の箇所に **OpenMP の指示を追加** し, 内部点を更新する二重ループを並列化せよ。

- C++: 二重ループの直前に `#pragma omp parallel for collapse(2) reduction(max:diff)` を1行加える。
- Fortran: 二重ループを `!$omp parallel do collapse(2) private(v,d) reduction(max:diff)` と `!$omp end parallel do` で囲む。

ポイント:

- `collapse(2)` で内側2重ループをまとめて並列化する (外側だけより多くのスレッドに仕事が行き渡る)。
- 各点の更新量の**最大値**が欲しいので `reduction(max:diff)` を使う (総和の `+` ではなく `max`)。
- 配列 `u` (現在) と `unew` (次) を毎反復ポインタで入れ替えている (コピーしない)。境界はずっと固定。

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore heat2d.cpp -o heat2d.exe

# Fortran
nvfortran -fast -mp=multicore heat2d.f90 -o heat2d.exe
```

第1引数で格子サイズ `L`, 第2引数で収束しきい値 `tol`, 第3引数で最大反復数。

```
OMP_NUM_THREADS=4 ./heat2d.exe 129 1e-6 1000000
```

## 期待される結果

上端だけ 100, 他3辺 0 の正方形板では, **対称性から中心温度はちょうど 25** になる (同じ問題を4回 90度回転して重ねると全周 100 = 内部 100 になり, 中心は 100/4 = 25)。

```
L=129, iters=31240, 最終残差=1.00e-06, 中心温度=24.9967 (理論 25.0)
```

- ヤコビ法は収束が遅い。`tol` を小さくする (例: 1e-4 → 1e-6) ほど中心温度が 25 に近づくが反復数も増える。
- `OMP_NUM_THREADS` を増やすと反復1回あたりが速くなる (台数効果)。`collapse(2)` の有無や `L` を変えて速さを比べてみよ。
- 解いている連立一次方程式は 2次元ラプラシアンである。同じ行列は `17_linalg` の CG法 (解く) やべき乗法 (固有振動モード) でも登場する。



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ heat2d.cpp
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <omp.h>

/* 2D 定常熱伝導 (ラプラス方程式) をヤコビ反復で解く。
   L×L の板。上端(行0)の温度を 100, 残り3辺を 0 に固定し, 内部が定常分布に
   落ち着くまで反復する。各内部点を上下左右4点の平均で更新する (5点ステンシル)。 */
int main(int argc, char ** argv) {
  int    L       = (argc > 1 ? atoi(argv[1]) : 129);     /* 一辺の格子点数 */
  double tol     = (argc > 2 ? atof(argv[2]) : 1e-6);    /* 収束判定 (最大更新量) */
  int    maxiter = (argc > 3 ? atoi(argv[3]) : 1000000);

  int n = L * L;
  double * u    = (double *)malloc(sizeof(double) * n);
  double * unew = (double *)malloc(sizeof(double) * n);
  /* 初期化: 上端(行0)=100, 他=0。境界はずっと固定なので両配列に同じ値を入れておく。 */
  for (int i = 0; i < L; i++)
    for (int j = 0; j < L; j++) {
      double val = (i == 0) ? 100.0 : 0.0;
      u[i * L + j] = val;
      unew[i * L + j] = val;
    }

  int iter;
  double diff = 0.0;
  double t0 = omp_get_wtime();
  for (iter = 0; iter < maxiter; iter++) {
    diff = 0.0;   /* この反復での最大更新量 */
    /* 内部の各点 (i,j) を上下左右の平均で更新し, 更新量の最大値を求める。 */
    // TODO: 内側の二重ループを #pragma omp parallel for collapse(2) reduction(max:diff) で並列化せよ.
    for (int i = 1; i < L - 1; i++) {
      for (int j = 1; j < L - 1; j++) {
        double v = 0.25 * (u[(i - 1) * L + j] + u[(i + 1) * L + j]
                         + u[i * L + (j - 1)] + u[i * L + (j + 1)]);
        double d = fabs(v - u[i * L + j]);
        if (d > diff) diff = d;
        unew[i * L + j] = v;
      }
    }
    /* u と unew を入れ替える (コピーせずポインタを差し替えるだけ) */
    double * tmp = u; u = unew; unew = tmp;
    if (diff < tol) break;
  }
  double elapsed = omp_get_wtime() - t0;

  printf("L=%d, iters=%d, 最終残差=%.2e, 中心温度=%.4f (理論 25.0)\n",
         L, iter + 1, diff, u[(L / 2) * L + (L / 2)]);
  printf("elapsed = %.3f sec\n", elapsed);
  free(u); free(unew);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore heat2d.cpp -o heat2d_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./heat2d_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ heat2d.f90
! 2D 定常熱伝導 (ラプラス方程式) をヤコビ反復で解く。
! L×L の板。上端(行0)=100, 残り3辺=0 に固定し, 内部が定常分布に落ち着くまで反復。
! 各内部点を上下左右4点の平均で更新する (5点ステンシル)。
program heat2d
  use omp_lib
  implicit none
  integer :: L, maxiter, iter, i, j
  real(8) :: tol, diff, val, v, d, t0, elapsed
  real(8), allocatable, target :: a(:,:), b(:,:)
  real(8), pointer :: u(:,:), unew(:,:), tmp(:,:)
  character(len=32) :: arg
  L = 129; tol = 1d-6; maxiter = 1000000
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) L
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) tol
  end if
  if (command_argument_count() >= 3) then
     call get_command_argument(3, arg); read (arg, *) maxiter
  end if

  allocate(a(0:L-1, 0:L-1), b(0:L-1, 0:L-1))
  ! 初期化: 上端(行0)=100, 他=0。境界はずっと固定なので両配列に同じ値を入れておく。
  do j = 0, L - 1
     do i = 0, L - 1
        val = merge(100.0d0, 0.0d0, i == 0)
        a(i, j) = val
        b(i, j) = val
     end do
  end do
  u => a; unew => b

  t0 = omp_get_wtime()
  do iter = 1, maxiter
     diff = 0.0d0   ! この反復での最大更新量
     ! 内部の各点 (i,j) を上下左右の平均で更新し, 更新量の最大値を求める。
     ! TODO: 内側の二重ループを !$omp parallel do collapse(2) private(v,d) reduction(max:diff) で並列化せよ.
     do j = 1, L - 2
        do i = 1, L - 2
           v = 0.25d0 * (u(i-1,j) + u(i+1,j) + u(i,j-1) + u(i,j+1))
           d = abs(v - u(i,j))
           if (d > diff) diff = d
           unew(i,j) = v
        end do
     end do
     ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do).
     ! u と unew を入れ替える (コピーせずポインタを差し替えるだけ)
     tmp => u; u => unew; unew => tmp
     if (diff < tol) exit
  end do
  elapsed = omp_get_wtime() - t0

  print "(a,i0,a,i0,a,es9.2,a,f0.4,a)", &
       "L=", L, ", iters=", min(iter, maxiter), ", 最終残差=", diff, &
       ", 中心温度=", u(L/2, L/2), " (理論 25.0)"
  print "(a,f0.3,a)", "elapsed = ", elapsed, " sec"
end program heat2d

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore heat2d.f90 -o heat2d_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./heat2d_f90.exe


# 4. 発展目標 (できる範囲で挑戦)
- この問題の基本は **マルチコア並列化** (`#pragma omp parallel for` / `!$omp parallel do` など)。まずはここまでを目指す。
- 余力があれば次にも挑戦:
  - **GPUで並列化**: コンパイルを `-mp=gpu` にして, 重いループに `#pragma omp target teams distribute parallel for` (+ 必要に応じて `map`) を付ける。
  - **SIMD化** (11/12章): 内側ループに `#pragma omp simd`, またはベクトル型。 **ILP向上** (13章): ベクトル長 `nl` の調整。
- どこまで速くできるか挑戦し, その効果を下の「性能を比べる」で可視化しよう。

# 5. 性能を比べる (任意)
- 各プログラムは主計算部分の所要時間を `elapsed = ... sec` の形で表示する。
- 構成を変えて (スレッド数, SIMDの有無, GPU など) 実行し, 得られた **経過秒** を下の `DATA` に「ラベルと秒」で並べて実行すると, 棒グラフと「基準(先頭)比のスピードアップ」が出る。
- 試した構成だけ書けばよい (`#` で始まる行は無視)。


In [ ]:
import matplotlib.pyplot as plt

# 各構成の (ラベル, 経過秒) を並べる。先頭の行を基準(スピードアップ=1)にする。
# 自分が実際に試した構成の数値に書き換える。
DATA = [
    ("serial",    1.00),
    ("omp-8",     0.20),
    # ("simd",      0.50),
    # ("simd+omp",  0.07),
    # ("gpu",       0.05),
]

labels = [d[0] for d in DATA]
secs   = [float(d[1]) for d in DATA]
speed  = [secs[0] / s for s in secs]                 # 先頭(基準)比のスピードアップ
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(labels, secs)
ax[0].set_ylabel("elapsed (s)")
ax[0].set_title("time (lower = faster)")
ax[1].bar(labels, speed)
ax[1].set_ylabel("speedup vs " + labels[0])
ax[1].set_title("speedup (higher = faster)")
for a in ax:
    a.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


# 6. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:heat2d.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 6-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 6-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 6-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:heat2d.cpp}

コマンドと実行結果:
{bash[-1]}

## 6-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:heat2d.cpp}